# PA4 — Document Analyst

Development & testing notebook for Tasks **1.7**, **2.4**, and **3.2**.

Fill `.env` before running. Deploy the endpoint (Task 2.2–2.3) before Part 2 / Part 3 cells.

## Setup

In [ ]:
import os
import time

from dotenv import load_dotenv

load_dotenv()

DATABRICKS_HOST = os.environ["DATABRICKS_HOST"].rstrip("/")
DATABRICKS_TOKEN = os.environ["DATABRICKS_TOKEN"]
ENDPOINT_NAME = os.environ.get("SERVING_ENDPOINT_NAME", "document-analyst")

print("HOST:", DATABRICKS_HOST)
print("ENDPOINT:", ENDPOINT_NAME)

---
## Task 1.7 — Wire the Full Graph

1. Visualize the compiled graph  
2. Retrieval-only query  
3. Computation-only query  
4. Combined query  
5. Step-by-step execution trace for the combined query

In [ ]:
from agent.graph import build_graph

graph = build_graph()
graph

In [ ]:
# 1) Visualize the compiled graph
try:
    from IPython.display import Image, display

    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("PNG render unavailable; mermaid source instead:")
    print(graph.get_graph().draw_mermaid())
    print("(", exc, ")")

In [ ]:
# 2) Retrieval-only query
q_retrieval = "What was the net income in 2023?"
local_retrieval = graph.invoke({"messages": [{"role": "user", "content": q_retrieval}]})
print("PLAN:", local_retrieval.get("plan"))
print("STEP RESULTS:", local_retrieval.get("step_results"))
print("ANSWER:", local_retrieval["messages"][-1].content)

In [ ]:
# 3) Computation-only query
q_compute = "What is 15% of 2.4 billion?"
local_compute = graph.invoke({"messages": [{"role": "user", "content": q_compute}]})
print("PLAN:", local_compute.get("plan"))
print("STEP RESULTS:", local_compute.get("step_results"))
print("ANSWER:", local_compute["messages"][-1].content)

In [ ]:
# 4) Combined query
q_combined = "What was the revenue in 2023, and what would a 10% increase look like?"
local_combined = graph.invoke({"messages": [{"role": "user", "content": q_combined}]})
print("PLAN:", local_combined.get("plan"))
print("STEP RESULTS:", local_combined.get("step_results"))
print("ANSWER:", local_combined["messages"][-1].content)

In [ ]:
# 5) Step-by-step execution trace for the combined query
print("=== Combined query execution trace ===")
for event in graph.stream(
    {"messages": [{"role": "user", "content": q_combined}]},
    stream_mode="updates",
):
    for node_name, update in event.items():
        print(f"\n[{node_name}]")
        if isinstance(update, dict):
            for k in ("plan", "current_step_index", "next_agent", "step_results", "final_answer"):
                if k in update:
                    print(f"  {k}: {update[k]}")
            if "messages" in update and update["messages"]:
                last = update["messages"][-1]
                content = getattr(last, "content", last)
                print(f"  messages[-1]: {content}")

---
## Task 2.4 — Test the Deployed Endpoint

Requires a READY serving endpoint (`SERVING_ENDPOINT_NAME` in `.env`).

In [ ]:
# 1) Call the endpoint using curl and show the raw response
import subprocess
import json

curl_payload = json.dumps(
    {"messages": [{"role": "user", "content": "What was the net income in 2023?"}]}
)
curl_cmd = [
    "curl",
    "-s",
    "-X",
    "POST",
    f"{DATABRICKS_HOST}/serving-endpoints/{ENDPOINT_NAME}/invocations",
    "-H",
    f"Authorization: Bearer {DATABRICKS_TOKEN}",
    "-H",
    "Content-Type: application/json",
    "-d",
    curl_payload,
]
raw = subprocess.check_output(curl_cmd, text=True)
print(raw)

In [ ]:
# 2) Call the endpoint using the OpenAI Python SDK and show the parsed response
import openai

client = openai.OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints",
)

response = client.chat.completions.create(
    model=ENDPOINT_NAME,
    messages=[{"role": "user", "content": "What was the net income in 2023?"}],
)
print(response)
print("\nParsed content:")
print(response.choices[0].message.content)

In [ ]:
# 3) Run the same 3 test queries from Task 1.7 against the deployed endpoint
deployed_queries = [q_retrieval, q_compute, q_combined]
deployed_answers = {}

for q in deployed_queries:
    resp = client.chat.completions.create(
        model=ENDPOINT_NAME,
        messages=[{"role": "user", "content": q}],
    )
    answer = resp.choices[0].message.content
    deployed_answers[q] = answer
    print("Q:", q)
    print("A:", answer)
    print("---")

In [ ]:
# 4) Compare local vs. deployed responses
local_answers = {
    q_retrieval: local_retrieval["messages"][-1].content,
    q_compute: local_compute["messages"][-1].content,
    q_combined: local_combined["messages"][-1].content,
}

for q in deployed_queries:
    print("Query:", q)
    print("LOCAL   :", local_answers[q])
    print("DEPLOYED:", deployed_answers[q])
    print("Identical:", local_answers[q].strip() == deployed_answers[q].strip())
    print("---")

print(
    "Note: Answers may differ due to LLM sampling, cold vs warm containers, "
    "or slight prompt/runtime differences even with temperature=0."
)

In [ ]:
# 5) Measure and report: latency per request (cold start vs. warm)
latency_query = "What was the net income in 2023?"

def timed_call(label):
    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model=ENDPOINT_NAME,
        messages=[{"role": "user", "content": latency_query}],
    )
    elapsed = time.perf_counter() - t0
    print(f"{label}: {elapsed:.2f}s")
    print("  content:", (resp.choices[0].message.content or "")[:200], "...")
    return elapsed

cold_s = timed_call("Cold / first request")
warm_s = timed_call("Warm / second request")
print(f"\nCold: {cold_s:.2f}s | Warm: {warm_s:.2f}s | Delta: {cold_s - warm_s:.2f}s")

---
## Task 3.2 — Demonstrate the Client

1. Import and instantiate `DocumentAnalystClient`  
2. `health_check()` → assert `True`  
3. `ask()`  
4. `ask_streaming()`  
5. Simulate timeout (`timeout=0.001`)  
6. Simulate unavailable endpoint / retry behavior

In [ ]:
# 1) Import and instantiate DocumentAnalystClient
from client.sdk import AnalystClientError, DocumentAnalystClient

analyst = DocumentAnalystClient(endpoint_name=ENDPOINT_NAME)
analyst

In [ ]:
# 2) Run health_check() and assert it returns True
ok = analyst.health_check()
print("health_check:", ok)
assert ok is True, "Endpoint is not READY"

In [ ]:
# 3) Call ask() with a simple query and display the result
answer = analyst.ask("What was the net income in 2023?")
print(answer)

In [ ]:
# 4) Call ask_streaming() and print chunks as they arrive
print("Streaming chunks:")
for chunk in analyst.ask_streaming("What was the net income in 2023?"):
    print(repr(chunk))

In [ ]:
# 5) Simulate a timeout (set timeout=0.001) and show the error handling
slow = DocumentAnalystClient(endpoint_name=ENDPOINT_NAME, timeout=0.001)
try:
    slow.ask("What was the net income in 2023?")
except TimeoutError as exc:
    print("Caught TimeoutError:", exc)

In [ ]:
# 6) Simulate the endpoint being unavailable and show retry behavior
bad = DocumentAnalystClient(
    endpoint_name="does-not-exist-endpoint",
    max_retries=2,
    timeout=30.0,
)
t0 = time.perf_counter()
try:
    bad.ask("ping")
except (AnalystClientError, TimeoutError, Exception) as exc:
    elapsed = time.perf_counter() - t0
    print(f"Caught {type(exc).__name__} after {elapsed:.2f}s: {exc}")
    if isinstance(exc, AnalystClientError):
        print("status_code:", exc.status_code)
        print("request_id:", exc.request_id)

### Optional: force 429/503-style retries

If you need to demo exponential backoff explicitly, point the client at a URL that returns 503
(or temporarily stop/scale the endpoint) and watch `ask()` retry with backoff.